This notebook aims at comparing two tables of flux and fluxerr

In [1]:
# load tables
import numpy as np
import pandas as pd

dfsuper = pd.read_csv('SUPER_catalog.txt', sep='\t', index_col=0, na_values="NaN")
dfcat = pd.read_csv('Flux_table_Abell2744.txt', sep='\t', header=None, index_col=0, na_values="NaN")
dfcat.columns = ['id', 'ra', 'dec', 'f_f606w', 'f_f277w', 'f_f356w','f_f444w',
                 'e_f606w', 'e_f277w', 'e_f356w','e_f444w']
print(dfsuper)

            id             x             y        ra        dec  \
0          1.0   4170.015070      0.803660  3.624497 -30.467330   
1          2.0   4191.496535      2.501775  3.624220 -30.467312   
2          3.0   5173.648912      1.553996  3.611559 -30.467325   
3          4.0   5249.669770      2.675741  3.610579 -30.467313   
4          5.0   3820.239014      1.533098  3.629006 -30.467321   
...        ...           ...           ...       ...        ...   
74015  74016.0  13486.195668  15937.434567  3.504552 -30.290236   
74016  74017.0  13533.352425  15963.635268  3.503945 -30.289944   
74017  74018.0  13563.461010  15968.209260  3.503558 -30.289893   
74018  74019.0  13513.656173  15999.087055  3.504199 -30.289551   
74019  74020.0  13476.758572  16003.528709  3.504674 -30.289502   

       kron_radius_circ    f_f606w      f_f277w  f_f356w  f_f444w   e_f606w  \
0                   NaN        NaN          NaN      NaN      NaN       NaN   
1                   NaN        NaN   

In [3]:
# load segmentation map data
import matplotlib.pyplot as plt
from astropy.io import fits
import numpy as np
from astropy.wcs import WCS

source_map, im_header = fits.getdata('segmentation_map.fits', header=True)
im_wcs = WCS(im_header)
print(source_map.shape)


(16384, 21504)


In [5]:
# compare SUPER catalog fluxes with sourcefinder flux

c = 0

super_catalog = []
source_finder = []

for i in range(len(dfsuper)):
    if dfsuper.use_phot[i] == 1:
        xp, yp = dfsuper.x[i], dfsuper.y[i]
        sid = source_map[int(yp),int(xp)]
        if (sid != 0) and (sid <= 1500):
            # print("SUPER object #", dfsuper.id[i], " corresponds to source id #", sid)
            # Header
            # print("-" * 70)
            # print(f"{'Cat':<8} {'ID':<6} {'F606W':>12} {'F277W':>12} {'F356W':>12} {'F444W':>12}")
            # print("-" * 70)

            # SUPER row
            # print(f"{'SUPER':<8} {dfsuper.id[i]:<6} "
            #       f"{dfsuper.f_f606w[i]:>12.5f} "   
            #       f"{dfsuper.f_f277w[i]:>12.5f} "
            #       f"{dfsuper.f_f356w[i]:>12.5f} "
            #       f"{dfsuper.f_f444w[i]:>12.5f}")
            super_catalog.append(dfsuper.f_f606w[i])
            super_catalog.append(dfsuper.f_f277w[i])
            super_catalog.append(dfsuper.f_f356w[i])
            super_catalog.append(dfsuper.f_f444w[i])
            
            # Source row
            # print(f"{'Source':<8} {dfcat.id[sid-1]:<6} "
            #       f"{dfcat.f_f606w[sid-1]:>12.5f} "
            #       f"{dfcat.f_f277w[sid-1]:>12.5f} "
            #       f"{dfcat.f_f356w[sid-1]:>12.5f} "
            #       f"{dfcat.f_f444w[sid-1]:>12.5f}")
            source_finder.append(dfcat.f_f606w[sid-1])
            source_finder.append(dfcat.f_f277w[sid-1])
            source_finder.append(dfcat.f_f356w[sid-1])
            source_finder.append(dfcat.f_f444w[sid-1])
            
            # print(" ")
            # print(" ")
            c=c+1
print("Number of SUPER objects among first 1500 detected sources = ", c)
print("SUPER catalog fluxes = ", len(super_catalog))
print("SourceFinder fluxes = ", len(source_finder))

Number of SUPER objects among first 1500 detected sources =  1049
SUPER catalog fluxes =  4196
SourceFinder fluxes =  4196


In [7]:
from astropy.table import Table
import numpy as np
import pandas as pd
col_names = ["super_flux", "finder_flux"]
array = np.column_stack((super_catalog,source_finder))
print(array.shape)
dfc = pd.DataFrame(array, columns=col_names)

# export results to .txt file

from astropy.table import Table

# Save the flux data to Excel
dfc.to_excel('Flux_correlation.xlsx', index=False)
# Save the flux data to .txt file for BAGPIPES
dfc.to_csv('Flux_correlation.txt', sep='\t', na_rep="NaN", index=True, header=False)

(4196, 2)
